# Deep Q-Network (DQN)

## 1. Konsep Dasar DQN

**Deep Q-Network (DQN)** merupakan pengembangan dari Q-Learning yang menggunakan **Neural Network** untuk memperkirakan nilai Q.

Pada Q-Learning biasa, nilai Q disimpan dalam **Q-Table**. Jika jumlah state sangat banyak atau berupa data kontinyu seperti posisi, kecepatan, atau sensor, Q-Table dapat menjadi sangat besar.

DQN mengganti Q-Table dengan **Q-Network** yang menerima state sebagai input dan menghasilkan nilai Q untuk setiap action.

Pada kode ini, Q-Network memiliki:

* Input: `state_dim`
* Hidden layer: 64 neuron + ReLU
* Hidden layer: 64 neuron + ReLU
* Output: `action_dim`

---

## 2. Dua Komponen Utama DQN

### 1. Replay Buffer

Replay Buffer menyimpan pengalaman agent dalam bentuk:

```text
(state, action, reward, next_state, done)
```

Pada kode, kapasitasnya adalah **10.000 pengalaman**.

Saat training, sebanyak **64 pengalaman** diambil secara acak dari buffer:

```python
batch = random.sample(self.buffer, batch_size)
```

Pengambilan secara acak membantu mengurangi korelasi antar pengalaman yang berurutan.

### 2. Target Network

DQN menggunakan dua Neural Network:

* **Policy Network (`policy_net`)** → digunakan untuk memilih action dan diperbarui setiap training.
* **Target Network (`target_net`)** → digunakan untuk menghitung target Q-value dan diperbarui secara berkala.

Pada kode, Target Network diperbarui setiap **10 episode**:

```python
target_net.load_state_dict(policy_net.state_dict())
```

Tujuannya adalah membuat target pembelajaran lebih stabil.

---

## 3. Epsilon-Greedy

DQN menggunakan epsilon-greedy untuk menyeimbangkan eksplorasi dan eksploitasi.

Parameter pada kode:

```text
epsilon awal = 1.0
epsilon minimum = 0.01
epsilon decay = 0.995
```

Jika melakukan eksplorasi, agent memilih action secara acak.

Jika melakukan eksploitasi, agent memilih action dengan nilai Q tertinggi dari Policy Network.

Seiring training berjalan, nilai epsilon semakin kecil sehingga agent semakin sering memanfaatkan pengetahuan yang telah dipelajari.

---

## 4. Proses Training

Secara sederhana:

```text
State
  ↓
Policy Network
  ↓
Pilih Action
  ↓
Environment
  ↓
Reward + Next State
  ↓
Replay Buffer
  ↓
Random Batch
  ↓
Policy Network + Target Network
  ↓
Hitung Loss
  ↓
Update Policy Network
```

Target Q-value dihitung dengan:

$$
Q_{target} = r + \gamma \max_{a'} Q_{target}(s', a')
$$

Pada kode, `gamma = 0.99`.

Loss dihitung menggunakan **MSE Loss**, kemudian Policy Network diperbarui menggunakan optimizer **Adam**.

---

## 5. Inti DQN

**Q-Learning:**

```text
State + Action → Q-Table → Q-value
```

**DQN:**

```text
State → Neural Network → Q-value
```

DQN menggunakan **Replay Buffer** untuk mengambil pengalaman secara acak dan **Target Network** untuk membuat proses pembelajaran lebih stabil.


In [1]:
import random
import collections
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as env_lib

# 1. Definisi Neural Network untuk Q-Function
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(QNetwork, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

    def forward(self, x):
        return self.fc(x)

# 2. Replay Buffer
class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = collections.deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.FloatTensor(np.array(states)),
            torch.LongTensor(actions),
            torch.FloatTensor(rewards),
            torch.FloatTensor(np.array(next_states)),
            torch.FloatTensor(dones)
        )

    def __len__(self):
        return len(self.buffer)

# 3. Setup Environment dan Hyperparameter
env = env_lib.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

policy_net = QNetwork(state_dim, action_dim)
target_net = QNetwork(state_dim, action_dim)
target_net.load_state_dict(policy_net.state_dict()) # Samakan bobot awal

optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)
memory = ReplayBuffer(capacity=10000)

gamma = 0.99
batch_size = 64
epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.995
target_update_freq = 10

# 4. Training Loop
num_episodes = 200

for episode in range(num_episodes):
    state, _ = env.reset()
    total_reward = 0
    done = False

    while not done:
        # Epsilon-greedy action
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                state_t = torch.FloatTensor(state).unsqueeze(0)
                q_values = policy_net(state_t)
                action = q_values.argmax().item()

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        memory.push(state, action, reward, next_state, float(done))
        state = next_state
        total_reward += reward

        # Optimize model jika buffer sudah cukup isi
        if len(memory) >= batch_size:
            b_states, b_actions, b_rewards, b_next_states, b_dones = memory.sample(batch_size)

            # Q(s, a) dari policy net
            q_eval = policy_net(b_states).gather(1, b_actions.unsqueeze(1)).squeeze(1)

            # max a' Q_target(s', a') dari target net
            with torch.no_grad():
                max_next_q = target_net(b_next_states).max(1)[0]
                q_target = b_rewards + (gamma * max_next_q * (1 - b_dones))

            loss = nn.MSELoss()(q_eval, q_target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Decay epsilon
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    # Update Target Network berkala
    if episode % target_update_freq == 0:
        target_net.load_state_dict(policy_net.state_dict())

    if (episode + 1) % 20 == 0:
        print(f"Episode {episode + 1}/{num_episodes} - Total Reward: {total_reward} - Epsilon: {epsilon:.2f}")

env.close()

Episode 20/200 - Total Reward: 82.0 - Epsilon: 0.90
Episode 40/200 - Total Reward: 44.0 - Epsilon: 0.82
Episode 60/200 - Total Reward: 35.0 - Epsilon: 0.74
Episode 80/200 - Total Reward: 13.0 - Epsilon: 0.67
Episode 100/200 - Total Reward: 33.0 - Epsilon: 0.61
Episode 120/200 - Total Reward: 126.0 - Epsilon: 0.55
Episode 140/200 - Total Reward: 21.0 - Epsilon: 0.50
Episode 160/200 - Total Reward: 63.0 - Epsilon: 0.45
Episode 180/200 - Total Reward: 29.0 - Epsilon: 0.41
Episode 200/200 - Total Reward: 134.0 - Epsilon: 0.37
